In [26]:
import cx_Oracle
import itertools
import joblib
import shap
from tqdm import tqdm
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
# import missingno as msno

In [27]:
%matplotlib inline
sns.set_style('whitegrid')
from matplotlib import font_manager
# font_path='/usr/share/fonts/cjkuni-uming/uming.ttc'
font_path = '/usr/share/fonts/truetype/arphic/uming.ttc'
matplotlib.rcParams['font.family']=font_manager.FontProperties(fname=font_path).get_name()
matplotlib.rcParams['axes.unicode_minus']=False

In [28]:
import warnings
warnings.filterwarnings('ignore')

In [29]:
%matplotlib inline

### 现金流量表，季报对齐到每天

In [5]:
# 获取现金流量表中的季报数据
connection = cx_Oracle.connect('wind', 'wind', '10.6.60.114:1521/wind')
cursor = connection.cursor()
sql = "select * from ASHARECASHFLOWHIS where  STATEMENT_TYPE in (408001000,408005000,408027000,408028000,408036000,408045000)  order by REPORT_PERIOD,ACTUAL_ANN_DT"
cursor.execute(sql)
columns = [col[0] for col in cursor.description]
results = cursor.fetchall()
df1 = pd.DataFrame(results, columns=columns)
cursor.close()
connection.close()

In [30]:
df1.shape

(252266, 122)

In [31]:
df1.head()

,OBJECT_ID,S_INFO_WINDCODE,S_INFO_COMPCODE,ANN_DT,ACTUAL_ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,CRNCY_CODE,CASH_RECP_SG_AND_RS,RECP_TAX_RENDS,...,TOT_BAL_NETCASH_INC_UNDIR,S_DISMANTLE_CAPITAL_ADD_NET,IS_CALCULATION,SECURITIE_NETCASH_RECEIVED,OTHER_IMPAIR_LOSS_ASSETS,CREDIT_IMPAIRMENT_LOSS,RIGHT_USE_ASSETS_DEP,STATEMENT_TYPE_WIND,OPDATE,OPMODE
0,{F417E2D4-B2DA-43A7-8CBD-64BB0759B996},600015.SH,08M399B152,20070314,20060217,20051231,408001000,CNY,NaN,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2018-07-31 10:38:38,0
1,{E3791D76-86FA-4B6B-ABC1-AD174EC6A2AE},600016.SH,1600016,20070319,20060228,20051231,408001000,CNY,NaN,NaN,...,NaN,1.125393e+10,0.0,NaN,None,NaN,NaN,合并报表,2019-03-19 13:30:52,0
2,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,20060302,20051231,408001000,CNY,NaN,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
3,{96D6CF45-08F7-44BF-B19A-63E987CADBDC},000001.SZ,1000001,20070322,20060401,20051231,408001000,CNY,NaN,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2023-08-01 15:59:44,0
4,{3A52DC8F-D3E7-C747-E040-007F01001792},600036.SH,1600036,20060412,20060412,20051231,408001000,CNY,NaN,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2019-03-19 09:48:07,0


In [32]:
start_date=df1['REPORT_PERIOD'].min()

In [42]:
start_date

'20051231'

In [34]:
end_date='20250626'

In [35]:
#生成日期序列
date_range=pd.date_range(start=start_date,end=end_date,freq='D')
df_dates=pd.DataFrame({'date':date_range})

In [36]:
#格式化日期
df_dates['Date']=df_dates['date'].dt.strftime("%Y%m%d")
df_dates['Date']=pd.to_datetime(df_dates['Date'])

In [37]:
df_dates=df_dates['Date']

In [38]:
df_dates

0      2005-12-31
1      2006-01-01
2      2006-01-02
3      2006-01-03
4      2006-01-04
          ...    
7113   2025-06-22
7114   2025-06-23
7115   2025-06-24
7116   2025-06-25
7117   2025-06-26
Name: Date, Length: 7118, dtype: datetime64[ns]

In [14]:
df2=df1[df1['S_INFO_WINDCODE']=='600000.SH']
df2['ACTUAL_ANN_DT']=pd.to_datetime(df2['ACTUAL_ANN_DT'])

In [16]:
df2

,OBJECT_ID,S_INFO_WINDCODE,S_INFO_COMPCODE,ANN_DT,ACTUAL_ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,CRNCY_CODE,CASH_RECP_SG_AND_RS,RECP_TAX_RENDS,...,TOT_BAL_NETCASH_INC_UNDIR,S_DISMANTLE_CAPITAL_ADD_NET,IS_CALCULATION,SECURITIE_NETCASH_RECEIVED,OTHER_IMPAIR_LOSS_ASSETS,CREDIT_IMPAIRMENT_LOSS,RIGHT_USE_ASSETS_DEP,STATEMENT_TYPE_WIND,OPDATE,OPMODE
2,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,2006-03-02,20051231,408001000,CNY,NaN,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
1124,{47D2329D-DA03-3DF5-E040-007F010061A8},600000.SH,1600000,20070428,2006-04-29,20060331,408001000,CNY,NaN,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2024-10-08 22:38:07,0
1865,{F6C864C7-DCB2-4A9D-B219-54B506220FE7},600000.SH,1600000,20060812,2006-08-12,20060630,408001000,CNY,NaN,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2018-08-01 16:27:26,0
3608,{38B99680-27E0-4E7B-842F-0324EF11CD08},600000.SH,1600000,20061026,2006-10-27,20060930,408001000,CNY,NaN,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2024-10-08 22:35:24,0
4751,{6EA8B473-816C-40EB-96B2-28C7B579D3D4},600000.SH,1600000,20070324,2007-03-24,20061231,408001000,CNY,NaN,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2024-10-08 22:31:41,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
229678,{173A0B94-89F7-1B94-E063-2001C80A2150},600000.SH,1600000,20240430,2024-04-30,20240331,408001000,CNY,NaN,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2024-04-29 17:55:07,0
231054,{2007FDE7-1C11-8249-E063-1F01C80A5AD8},600000.SH,1600000,20240820,2024-08-20,20240630,408001000,CNY,NaN,NaN,...,NaN,NaN,0.0,NaN,None,3.254600e+10,NaN,合并报表,2024-08-19 20:36:41,0
241230,{25AEA42C-9BAB-BC81-E063-1F01C80A9CDD},600000.SH,1600000,20241031,2024-10-31,20240930,408001000,CNY,NaN,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2024-10-30 16:41:01,0
242172,{3166C4B7-0169-2011-E063-1F01C80ADDDE},600000.SH,1600000,20250329,2025-03-29,20241231,408001000,CNY,NaN,NaN,...,NaN,NaN,0.0,NaN,None,6.943700e+10,NaN,合并报表,2025-03-28 23:01:43,0


In [85]:
#合并两个数据框
df_merged=pd.merge_asof(df_dates,df2,left_on='Date',right_on='ACTUAL_ANN_DT',direction='backward')

In [86]:
df_merged=df_merged.dropna(subset=['S_INFO_WINDCODE'])

In [87]:
df_merged

,Date,OBJECT_ID,S_INFO_WINDCODE,S_INFO_COMPCODE,ANN_DT,ACTUAL_ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,CRNCY_CODE,CASH_RECP_SG_AND_RS,...,TOT_BAL_NETCASH_INC_UNDIR,S_DISMANTLE_CAPITAL_ADD_NET,IS_CALCULATION,SECURITIE_NETCASH_RECEIVED,OTHER_IMPAIR_LOSS_ASSETS,CREDIT_IMPAIRMENT_LOSS,RIGHT_USE_ASSETS_DEP,STATEMENT_TYPE_WIND,OPDATE,OPMODE
61,2006-03-02,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,2006-03-02,20051231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
62,2006-03-03,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,2006-03-02,20051231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
63,2006-03-04,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,2006-03-02,20051231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
64,2006-03-05,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,2006-03-02,20051231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
65,2006-03-06,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,2006-03-02,20051231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7113,2025-06-22,{33EA07C2-536A-19E3-E063-1F01C80AD435},600000.SH,1600000,20250430,2025-04-30,20250331,408001000,CNY,NaN,...,NaN,7.993000e+09,0.0,NaN,None,NaN,NaN,合并报表,2025-04-29 23:01:10,0
7114,2025-06-23,{33EA07C2-536A-19E3-E063-1F01C80AD435},600000.SH,1600000,20250430,2025-04-30,20250331,408001000,CNY,NaN,...,NaN,7.993000e+09,0.0,NaN,None,NaN,NaN,合并报表,2025-04-29 23:01:10,0
7115,2025-06-24,{33EA07C2-536A-19E3-E063-1F01C80AD435},600000.SH,1600000,20250430,2025-04-30,20250331,408001000,CNY,NaN,...,NaN,7.993000e+09,0.0,NaN,None,NaN,NaN,合并报表,2025-04-29 23:01:10,0
7116,2025-06-25,{33EA07C2-536A-19E3-E063-1F01C80AD435},600000.SH,1600000,20250430,2025-04-30,20250331,408001000,CNY,NaN,...,NaN,7.993000e+09,0.0,NaN,None,NaN,NaN,合并报表,2025-04-29 23:01:10,0


In [45]:
df_merged['ACTUAL_ANN_DT'].unique()

<DatetimeArray>
['2006-03-02 00:00:00', '2006-04-29 00:00:00', '2006-08-12 00:00:00',
 '2006-10-27 00:00:00', '2007-03-24 00:00:00', '2007-04-28 00:00:00',
 '2007-08-22 00:00:00', '2007-10-27 00:00:00', '2008-02-28 00:00:00',
 '2008-04-26 00:00:00', '2008-08-23 00:00:00', '2008-10-30 00:00:00',
 '2009-04-10 00:00:00', '2009-04-30 00:00:00', '2009-08-29 00:00:00',
 '2009-10-29 00:00:00', '2010-04-07 00:00:00', '2010-04-30 00:00:00',
 '2010-08-30 00:00:00', '2010-10-29 00:00:00', '2011-03-30 00:00:00',
 '2011-04-28 00:00:00', '2011-08-16 00:00:00', '2011-10-29 00:00:00',
 '2012-03-16 00:00:00', '2012-04-28 00:00:00', '2012-08-15 00:00:00',
 '2012-10-31 00:00:00', '2013-03-14 00:00:00', '2013-04-27 00:00:00',
 '2013-08-14 00:00:00', '2013-10-31 00:00:00', '2014-03-20 00:00:00',
 '2014-04-30 00:00:00', '2014-08-14 00:00:00', '2014-10-31 00:00:00',
 '2015-03-19 00:00:00', '2015-04-30 00:00:00', '2015-08-20 00:00:00',
 '2015-10-30 00:00:00', '2016-04-07 00:00:00', '2016-04-30 00:00:00',
 '20

#### 过去一年时间轴

In [21]:
# 获取现金流量表中的年报数据
connection = cx_Oracle.connect('wind', 'wind', '10.6.60.114:1521/wind')
cursor = connection.cursor()
sql = "select * from ASHARECASHFLOWHIS where  STATEMENT_TYPE in (408001000,408004000,408050000,408029000,408031000,408037000,408046000)  order by REPORT_PERIOD,ACTUAL_ANN_DT"
cursor.execute(sql)
columns = [col[0] for col in cursor.description]
results = cursor.fetchall()
df3 = pd.DataFrame(results, columns=columns)
cursor.close()
connection.close()

In [39]:
df3.shape

(488449, 122)

In [43]:
df4=df3[df3['S_INFO_WINDCODE']=='600000.SH']

In [114]:
df4

,OBJECT_ID,S_INFO_WINDCODE,S_INFO_COMPCODE,ANN_DT,ACTUAL_ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,CRNCY_CODE,CASH_RECP_SG_AND_RS,RECP_TAX_RENDS,...,TOT_BAL_NETCASH_INC_UNDIR,S_DISMANTLE_CAPITAL_ADD_NET,IS_CALCULATION,SECURITIE_NETCASH_RECEIVED,OTHER_IMPAIR_LOSS_ASSETS,CREDIT_IMPAIRMENT_LOSS,RIGHT_USE_ASSETS_DEP,STATEMENT_TYPE_WIND,OPDATE,OPMODE
0,{3A52DC8F-9F02-C747-E040-007F01001792},600000.SH,1600000,20020817,20020817,20010630,408004000,CNY,NaN,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表(调整),2024-10-08 22:37:07,0
2,{9A8A48D7-61EB-49F3-9B4F-248317DF6943},600000.SH,1600000,20060812,20060812,20050630,408004000,CNY,NaN,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表(调整),2018-08-01 16:27:53,0
5,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,20060302,20051231,408001000,CNY,NaN,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
12,{62578663-E788-4FD2-B660-D1EAD68672D3},600000.SH,1600000,20070324,20070324,20051231,408004000,CNY,NaN,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表(调整),2019-06-18 23:42:47,0
1127,{47D2329D-DA03-3DF5-E040-007F010061A8},600000.SH,1600000,20070428,20060429,20060331,408001000,CNY,NaN,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2024-10-08 22:38:07,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
466228,{33EA07C2-536E-19E3-E063-1F01C80AD435},600000.SH,1600000,20250430,20250430,20240331,408004000,CNY,NaN,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表(调整),2025-04-29 23:01:10,0
467223,{2007FDE7-1C11-8249-E063-1F01C80A5AD8},600000.SH,1600000,20240820,20240820,20240630,408001000,CNY,NaN,NaN,...,NaN,NaN,0.0,NaN,None,3.254600e+10,NaN,合并报表,2024-08-19 20:36:41,0
477318,{25AEA42C-9BAB-BC81-E063-1F01C80A9CDD},600000.SH,1600000,20241031,20241031,20240930,408001000,CNY,NaN,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2024-10-30 16:41:01,0
478130,{3166C4B7-0169-2011-E063-1F01C80ADDDE},600000.SH,1600000,20250329,20250329,20241231,408001000,CNY,NaN,NaN,...,NaN,NaN,0.0,NaN,None,6.943700e+10,NaN,合并报表,2025-03-28 23:01:43,0


In [235]:
# 2. 计算实际报告间隔（动态替代固定3个月）
df_merged['REPORT_PERIOD']=pd.to_datetime(df_merged['REPORT_PERIOD'])
df_quarterly = df_merged.drop_duplicates(subset='REPORT_PERIOD').copy()
df_quarterly['next_REPORT_PERIOD'] = df_quarterly['REPORT_PERIOD'].shift(-4)  # 获取下一次报告的实际日期
df_quarterly['actual_interval'] = (df_quarterly['next_REPORT_PERIOD'] - df_quarterly['REPORT_PERIOD']).dt.days


In [236]:
df_quarterly

,Date,OBJECT_ID,S_INFO_WINDCODE,S_INFO_COMPCODE,ANN_DT,ACTUAL_ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,CRNCY_CODE,CASH_RECP_SG_AND_RS,...,SECURITIE_NETCASH_RECEIVED,OTHER_IMPAIR_LOSS_ASSETS,CREDIT_IMPAIRMENT_LOSS,RIGHT_USE_ASSETS_DEP,STATEMENT_TYPE_WIND,OPDATE,OPMODE,IS_ADJUSTED,next_REPORT_PERIOD,actual_interval
61,2006-03-02,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,2006-03-02,2005-12-31,408001000,CNY,NaN,...,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0,False,2006-12-31,365.0
119,2006-04-29,{47D2329D-DA03-3DF5-E040-007F010061A8},600000.SH,1600000,20070428,2006-04-29,2006-03-31,408001000,CNY,NaN,...,NaN,None,NaN,NaN,合并报表,2024-10-08 22:38:07,0,False,2007-03-31,365.0
224,2006-08-12,{F6C864C7-DCB2-4A9D-B219-54B506220FE7},600000.SH,1600000,20060812,2006-08-12,2006-06-30,408001000,CNY,NaN,...,NaN,None,NaN,NaN,合并报表,2018-08-01 16:27:26,0,False,2007-06-30,365.0
300,2006-10-27,{38B99680-27E0-4E7B-842F-0324EF11CD08},600000.SH,1600000,20061026,2006-10-27,2006-09-30,408001000,CNY,NaN,...,NaN,None,NaN,NaN,合并报表,2024-10-08 22:35:24,0,False,2007-09-30,365.0
448,2007-03-24,{6EA8B473-816C-40EB-96B2-28C7B579D3D4},600000.SH,1600000,20070324,2007-03-24,2006-12-31,408001000,CNY,NaN,...,NaN,None,NaN,NaN,合并报表,2024-10-08 22:31:41,0,False,2007-12-31,365.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6695,2024-04-30,{173A0B94-89F7-1B94-E063-2001C80A2150},600000.SH,1600000,20240430,2024-04-30,2024-03-31,408001000,CNY,NaN,...,NaN,None,NaN,NaN,合并报表,2024-04-29 17:55:07,0,False,2025-03-31,365.0
6807,2024-08-20,{2007FDE7-1C11-8249-E063-1F01C80A5AD8},600000.SH,1600000,20240820,2024-08-20,2024-06-30,408001000,CNY,NaN,...,NaN,None,3.254600e+10,NaN,合并报表,2024-08-19 20:36:41,0,False,NaT,NaN
6879,2024-10-31,{25AEA42C-9BAB-BC81-E063-1F01C80A9CDD},600000.SH,1600000,20241031,2024-10-31,2024-09-30,408001000,CNY,NaN,...,NaN,None,NaN,NaN,合并报表,2024-10-30 16:41:01,0,False,NaT,NaN
7028,2025-03-29,{3166C4B7-0169-2011-E063-1F01C80ADDDE},600000.SH,1600000,20250329,2025-03-29,2024-12-31,408001000,CNY,NaN,...,NaN,None,6.943700e+10,NaN,合并报表,2025-03-28 23:01:43,0,False,NaT,NaN


In [237]:
df_yoy=df4.copy()


In [238]:
df_yoy['IS_ADJUSTED'] = (df_yoy['STATEMENT_TYPE'] == '408004000')
df_yoy['REPORT_PERIOD']=pd.to_datetime(df_yoy['REPORT_PERIOD'])
df_yoy['ACTUAL_ANN_DT']=pd.to_datetime(df_yoy['ACTUAL_ANN_DT'])


In [239]:
df_yoy

,OBJECT_ID,S_INFO_WINDCODE,S_INFO_COMPCODE,ANN_DT,ACTUAL_ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,CRNCY_CODE,CASH_RECP_SG_AND_RS,RECP_TAX_RENDS,...,S_DISMANTLE_CAPITAL_ADD_NET,IS_CALCULATION,SECURITIE_NETCASH_RECEIVED,OTHER_IMPAIR_LOSS_ASSETS,CREDIT_IMPAIRMENT_LOSS,RIGHT_USE_ASSETS_DEP,STATEMENT_TYPE_WIND,OPDATE,OPMODE,IS_ADJUSTED
0,{3A52DC8F-9F02-C747-E040-007F01001792},600000.SH,1600000,20020817,2002-08-17,2001-06-30,408004000,CNY,NaN,NaN,...,NaN,0.0,NaN,None,NaN,NaN,合并报表(调整),2024-10-08 22:37:07,0,True
2,{9A8A48D7-61EB-49F3-9B4F-248317DF6943},600000.SH,1600000,20060812,2006-08-12,2005-06-30,408004000,CNY,NaN,NaN,...,NaN,0.0,NaN,None,NaN,NaN,合并报表(调整),2018-08-01 16:27:53,0,True
5,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,2006-03-02,2005-12-31,408001000,CNY,NaN,NaN,...,NaN,0.0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0,False
12,{62578663-E788-4FD2-B660-D1EAD68672D3},600000.SH,1600000,20070324,2007-03-24,2005-12-31,408004000,CNY,NaN,NaN,...,NaN,0.0,NaN,None,NaN,NaN,合并报表(调整),2019-06-18 23:42:47,0,True
1127,{47D2329D-DA03-3DF5-E040-007F010061A8},600000.SH,1600000,20070428,2006-04-29,2006-03-31,408001000,CNY,NaN,NaN,...,NaN,0.0,NaN,None,NaN,NaN,合并报表,2024-10-08 22:38:07,0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
466228,{33EA07C2-536E-19E3-E063-1F01C80AD435},600000.SH,1600000,20250430,2025-04-30,2024-03-31,408004000,CNY,NaN,NaN,...,NaN,0.0,NaN,None,NaN,NaN,合并报表(调整),2025-04-29 23:01:10,0,True
467223,{2007FDE7-1C11-8249-E063-1F01C80A5AD8},600000.SH,1600000,20240820,2024-08-20,2024-06-30,408001000,CNY,NaN,NaN,...,NaN,0.0,NaN,None,3.254600e+10,NaN,合并报表,2024-08-19 20:36:41,0,False
477318,{25AEA42C-9BAB-BC81-E063-1F01C80A9CDD},600000.SH,1600000,20241031,2024-10-31,2024-09-30,408001000,CNY,NaN,NaN,...,NaN,0.0,NaN,None,NaN,NaN,合并报表,2024-10-30 16:41:01,0,False
478130,{3166C4B7-0169-2011-E063-1F01C80ADDDE},600000.SH,1600000,20250329,2025-03-29,2024-12-31,408001000,CNY,NaN,NaN,...,NaN,0.0,NaN,None,6.943700e+10,NaN,合并报表,2025-03-28 23:01:43,0,False


In [240]:
# 创建一个字典，用于存储每个报告期对应的下一个报告期的ACTUAL_ANN_DT
next_period_ann_dt = {}
for idx, row in df_quarterly.iterrows():
    if pd.notna(row['next_REPORT_PERIOD']):
        # 查找下一个报告期对应的行
        next_period_rows = df_quarterly[df_quarterly['REPORT_PERIOD'] == row['next_REPORT_PERIOD']]
        if not next_period_rows.empty:
            next_period_ann_dt[row['REPORT_PERIOD']] = next_period_rows.iloc[0]['ACTUAL_ANN_DT']
            

In [241]:
next_period_ann_dt

{Timestamp('2005-12-31 00:00:00'): Timestamp('2007-03-24 00:00:00'),
 Timestamp('2006-03-31 00:00:00'): Timestamp('2007-04-28 00:00:00'),
 Timestamp('2006-06-30 00:00:00'): Timestamp('2007-08-22 00:00:00'),
 Timestamp('2006-09-30 00:00:00'): Timestamp('2007-10-27 00:00:00'),
 Timestamp('2006-12-31 00:00:00'): Timestamp('2008-02-28 00:00:00'),
 Timestamp('2007-03-31 00:00:00'): Timestamp('2008-04-26 00:00:00'),
 Timestamp('2007-06-30 00:00:00'): Timestamp('2008-08-23 00:00:00'),
 Timestamp('2007-09-30 00:00:00'): Timestamp('2008-10-30 00:00:00'),
 Timestamp('2007-12-31 00:00:00'): Timestamp('2009-04-10 00:00:00'),
 Timestamp('2008-03-31 00:00:00'): Timestamp('2009-04-30 00:00:00'),
 Timestamp('2008-06-30 00:00:00'): Timestamp('2009-08-29 00:00:00'),
 Timestamp('2008-09-30 00:00:00'): Timestamp('2009-10-29 00:00:00'),
 Timestamp('2008-12-31 00:00:00'): Timestamp('2010-04-07 00:00:00'),
 Timestamp('2009-03-31 00:00:00'): Timestamp('2010-04-30 00:00:00'),
 Timestamp('2009-06-30 00:00:00'):

In [242]:
# 遍历df_quarterly中的每一行
result_df = df_quarterly.copy()
for idx, orig_row in df_quarterly.iterrows():
    report_period = orig_row['REPORT_PERIOD']
    # 查找df_yoy中对应报告期的所有行（可能包含原始数据和多次调整数据）
    adj_rows = df_yoy[(df_yoy['REPORT_PERIOD'] == report_period) & (df_yoy['IS_ADJUSTED'] == True)]
    if not adj_rows.empty:
        # 获取下一个报告期的公告日期
        next_ann_dt = next_period_ann_dt.get(report_period)
        if next_ann_dt is not None:
            # 筛选出公告日期大于下一个报告期公告日期的调整数据
            valid_adj_rows = adj_rows[adj_rows['ACTUAL_ANN_DT'] <= next_ann_dt]
            if not valid_adj_rows.empty:
                # 使用最新的有效调整数据
                latest_adj_row = valid_adj_rows.iloc[-1]
                # 更新结果DataFrame中的财务数据列
                financial_columns = [col for col in result_df.columns]
                
                for col in financial_columns:
                    if col in latest_adj_row:
                        result_df.at[idx, col] = latest_adj_row[col]



In [243]:
result_df

,Date,OBJECT_ID,S_INFO_WINDCODE,S_INFO_COMPCODE,ANN_DT,ACTUAL_ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,CRNCY_CODE,CASH_RECP_SG_AND_RS,...,SECURITIE_NETCASH_RECEIVED,OTHER_IMPAIR_LOSS_ASSETS,CREDIT_IMPAIRMENT_LOSS,RIGHT_USE_ASSETS_DEP,STATEMENT_TYPE_WIND,OPDATE,OPMODE,IS_ADJUSTED,next_REPORT_PERIOD,actual_interval
61,2006-03-02,{62578663-E788-4FD2-B660-D1EAD68672D3},600000.SH,1600000,20070324,2007-03-24,2005-12-31,408004000,CNY,NaN,...,NaN,None,NaN,NaN,合并报表(调整),2019-06-18 23:42:47,0,True,2006-12-31,365.0
119,2006-04-29,{3B9DF5F0-8B4D-4B2C-B20C-9537B2DB3380},600000.SH,1600000,20070428,2007-04-28,2006-03-31,408004000,CNY,NaN,...,NaN,None,NaN,NaN,合并报表(调整),2024-10-08 22:34:35,0,True,2007-03-31,365.0
224,2006-08-12,{7DF76AE7-D27D-4933-9DD5-68E3C0EF9127},600000.SH,1600000,20070822,2007-08-22,2006-06-30,408004000,CNY,NaN,...,NaN,None,NaN,NaN,合并报表(调整),2019-03-18 18:49:25,0,True,2007-06-30,365.0
300,2006-10-27,{4B30EB0F-BF47-46C5-A759-CE924C8CD6D2},600000.SH,1600000,20071027,2007-10-27,2006-09-30,408004000,CNY,NaN,...,NaN,None,NaN,NaN,合并报表(调整),2019-03-18 18:49:25,0,True,2007-09-30,365.0
448,2007-03-24,{E85DFC95-2AEE-4E6B-9AA9-EEE711A06649},600000.SH,1600000,20080228,2008-02-28,2006-12-31,408004000,CNY,NaN,...,NaN,None,NaN,NaN,合并报表(调整),2019-03-18 18:49:25,0,True,2007-12-31,365.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6695,2024-04-30,{33EA07C2-536E-19E3-E063-1F01C80AD435},600000.SH,1600000,20250430,2025-04-30,2024-03-31,408004000,CNY,NaN,...,NaN,None,NaN,NaN,合并报表(调整),2025-04-29 23:01:10,0,True,2025-03-31,365.0
6807,2024-08-20,{2007FDE7-1C11-8249-E063-1F01C80A5AD8},600000.SH,1600000,20240820,2024-08-20,2024-06-30,408001000,CNY,NaN,...,NaN,None,3.254600e+10,NaN,合并报表,2024-08-19 20:36:41,0,False,NaT,NaN
6879,2024-10-31,{25AEA42C-9BAB-BC81-E063-1F01C80A9CDD},600000.SH,1600000,20241031,2024-10-31,2024-09-30,408001000,CNY,NaN,...,NaN,None,NaN,NaN,合并报表,2024-10-30 16:41:01,0,False,NaT,NaN
7028,2025-03-29,{3166C4B7-0169-2011-E063-1F01C80ADDDE},600000.SH,1600000,20250329,2025-03-29,2024-12-31,408001000,CNY,NaN,...,NaN,None,6.943700e+10,NaN,合并报表,2025-03-28 23:01:43,0,False,NaT,NaN


In [244]:
# 3. 构建动态偏移映射表
period_map = result_df.set_index('REPORT_PERIOD')['next_REPORT_PERIOD'].to_dict()

# 4. 标记需要平移的列
non_financial_columns = ['Date', 'REPORT_PERIOD']
financial_columns = [col for col in df_merged.columns if col not in non_financial_columns]

# 5. 创建平移后的DataFrame
df_shifted = df_merged.copy()

for col in financial_columns:
    # 为每个财务值找到下一次报告期的值
    df_quarterly_shifted = result_df.copy()
    df_quarterly_shifted['mapped_REPORT_PERIOD'] = df_quarterly_shifted['REPORT_PERIOD'].map(period_map)
    
    # 动态匹配
    temp_df = pd.merge_asof(
        df_merged[['REPORT_PERIOD']].sort_values('REPORT_PERIOD'),
        df_quarterly_shifted[['mapped_REPORT_PERIOD', col]]
            .dropna()
            .sort_values('mapped_REPORT_PERIOD'),
        left_on='REPORT_PERIOD',
        right_on='mapped_REPORT_PERIOD',
        direction='backward'
    )
    df_shifted[col] = temp_df[col].values

In [245]:
df_shifted=df_shifted.dropna(subset=['S_INFO_WINDCODE'])

#构建逆向字典
reverse_period_map={v:k for k,v in period_map.items() if pd.notnull(v)}

#映射回原始值
df_shifted['REPORT_PERIOD']=df_shifted['REPORT_PERIOD'].map(reverse_period_map)
df_shifted

,Date,OBJECT_ID,S_INFO_WINDCODE,S_INFO_COMPCODE,ANN_DT,ACTUAL_ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,CRNCY_CODE,CASH_RECP_SG_AND_RS,...,S_DISMANTLE_CAPITAL_ADD_NET,IS_CALCULATION,SECURITIE_NETCASH_RECEIVED,OTHER_IMPAIR_LOSS_ASSETS,CREDIT_IMPAIRMENT_LOSS,RIGHT_USE_ASSETS_DEP,STATEMENT_TYPE_WIND,OPDATE,OPMODE,IS_ADJUSTED
448,2007-03-24,{62578663-E788-4FD2-B660-D1EAD68672D3},600000.SH,1600000,20070324,2007-03-24,2005-12-31,408004000,CNY,NaN,...,NaN,0.0,NaN,NaN,NaN,NaN,合并报表(调整),2019-06-18 23:42:47,0,True
449,2007-03-25,{62578663-E788-4FD2-B660-D1EAD68672D3},600000.SH,1600000,20070324,2007-03-24,2005-12-31,408004000,CNY,NaN,...,NaN,0.0,NaN,NaN,NaN,NaN,合并报表(调整),2019-06-18 23:42:47,0,True
450,2007-03-26,{62578663-E788-4FD2-B660-D1EAD68672D3},600000.SH,1600000,20070324,2007-03-24,2005-12-31,408004000,CNY,NaN,...,NaN,0.0,NaN,NaN,NaN,NaN,合并报表(调整),2019-06-18 23:42:47,0,True
451,2007-03-27,{62578663-E788-4FD2-B660-D1EAD68672D3},600000.SH,1600000,20070324,2007-03-24,2005-12-31,408004000,CNY,NaN,...,NaN,0.0,NaN,NaN,NaN,NaN,合并报表(调整),2019-06-18 23:42:47,0,True
452,2007-03-28,{62578663-E788-4FD2-B660-D1EAD68672D3},600000.SH,1600000,20070324,2007-03-24,2005-12-31,408004000,CNY,NaN,...,NaN,0.0,NaN,NaN,NaN,NaN,合并报表(调整),2019-06-18 23:42:47,0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7113,2025-06-22,{33EA07C2-536E-19E3-E063-1F01C80AD435},600000.SH,1600000,20250430,2025-04-30,2024-03-31,408004000,CNY,NaN,...,1.365670e+11,0.0,NaN,NaN,3.843800e+10,NaN,合并报表(调整),2025-04-29 23:01:10,0,True
7114,2025-06-23,{33EA07C2-536E-19E3-E063-1F01C80AD435},600000.SH,1600000,20250430,2025-04-30,2024-03-31,408004000,CNY,NaN,...,1.365670e+11,0.0,NaN,NaN,3.843800e+10,NaN,合并报表(调整),2025-04-29 23:01:10,0,True
7115,2025-06-24,{33EA07C2-536E-19E3-E063-1F01C80AD435},600000.SH,1600000,20250430,2025-04-30,2024-03-31,408004000,CNY,NaN,...,1.365670e+11,0.0,NaN,NaN,3.843800e+10,NaN,合并报表(调整),2025-04-29 23:01:10,0,True
7116,2025-06-25,{33EA07C2-536E-19E3-E063-1F01C80AD435},600000.SH,1600000,20250430,2025-04-30,2024-03-31,408004000,CNY,NaN,...,1.365670e+11,0.0,NaN,NaN,3.843800e+10,NaN,合并报表(调整),2025-04-29 23:01:10,0,True


In [247]:
df_quarterly_shifted

,Date,OBJECT_ID,S_INFO_WINDCODE,S_INFO_COMPCODE,ANN_DT,ACTUAL_ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,CRNCY_CODE,CASH_RECP_SG_AND_RS,...,OTHER_IMPAIR_LOSS_ASSETS,CREDIT_IMPAIRMENT_LOSS,RIGHT_USE_ASSETS_DEP,STATEMENT_TYPE_WIND,OPDATE,OPMODE,IS_ADJUSTED,next_REPORT_PERIOD,actual_interval,mapped_REPORT_PERIOD
61,2006-03-02,{62578663-E788-4FD2-B660-D1EAD68672D3},600000.SH,1600000,20070324,2007-03-24,2005-12-31,408004000,CNY,NaN,...,None,NaN,NaN,合并报表(调整),2019-06-18 23:42:47,0,True,2006-12-31,365.0,2006-12-31
119,2006-04-29,{3B9DF5F0-8B4D-4B2C-B20C-9537B2DB3380},600000.SH,1600000,20070428,2007-04-28,2006-03-31,408004000,CNY,NaN,...,None,NaN,NaN,合并报表(调整),2024-10-08 22:34:35,0,True,2007-03-31,365.0,2007-03-31
224,2006-08-12,{7DF76AE7-D27D-4933-9DD5-68E3C0EF9127},600000.SH,1600000,20070822,2007-08-22,2006-06-30,408004000,CNY,NaN,...,None,NaN,NaN,合并报表(调整),2019-03-18 18:49:25,0,True,2007-06-30,365.0,2007-06-30
300,2006-10-27,{4B30EB0F-BF47-46C5-A759-CE924C8CD6D2},600000.SH,1600000,20071027,2007-10-27,2006-09-30,408004000,CNY,NaN,...,None,NaN,NaN,合并报表(调整),2019-03-18 18:49:25,0,True,2007-09-30,365.0,2007-09-30
448,2007-03-24,{E85DFC95-2AEE-4E6B-9AA9-EEE711A06649},600000.SH,1600000,20080228,2008-02-28,2006-12-31,408004000,CNY,NaN,...,None,NaN,NaN,合并报表(调整),2019-03-18 18:49:25,0,True,2007-12-31,365.0,2007-12-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6695,2024-04-30,{33EA07C2-536E-19E3-E063-1F01C80AD435},600000.SH,1600000,20250430,2025-04-30,2024-03-31,408004000,CNY,NaN,...,None,NaN,NaN,合并报表(调整),2025-04-29 23:01:10,0,True,2025-03-31,365.0,2025-03-31
6807,2024-08-20,{2007FDE7-1C11-8249-E063-1F01C80A5AD8},600000.SH,1600000,20240820,2024-08-20,2024-06-30,408001000,CNY,NaN,...,None,3.254600e+10,NaN,合并报表,2024-08-19 20:36:41,0,False,NaT,NaN,NaT
6879,2024-10-31,{25AEA42C-9BAB-BC81-E063-1F01C80A9CDD},600000.SH,1600000,20241031,2024-10-31,2024-09-30,408001000,CNY,NaN,...,None,NaN,NaN,合并报表,2024-10-30 16:41:01,0,False,NaT,NaN,NaT
7028,2025-03-29,{3166C4B7-0169-2011-E063-1F01C80ADDDE},600000.SH,1600000,20250329,2025-03-29,2024-12-31,408001000,CNY,NaN,...,None,6.943700e+10,NaN,合并报表,2025-03-28 23:01:43,0,False,NaT,NaN,NaT
